# Benchmark M1–M6 — Échantillon (Configuration B1 : LLM seul, zero-shot)

**Modèle testé :** Gemma 4 31B (via vLLM, API OpenAI-compatible)  
**Modules couverts :** M1 (Principe QA), M2 (Applied QA), M4 (QCM), M6 (Interprétation d'arrêt)  
**Modules skippés :** M3 (multi-saut, nécessite KG v1), M5 (temporalité, nécessite versioning)  

---
**Prérequis :**
1. Serveur vLLM démarré via `setup_cluster.sh`
2. Dossier `cluster_data/` présent (généré par `prepare_cluster_data.py` en local)
3. Ajuster les variables de la cellule **Configuration** ci-dessous

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CONFIGURATION — Adapter avant de lancer
# ═══════════════════════════════════════════════════════════════════════

VLLM_BASE_URL = "http://localhost:8000/v1"   # URL du serveur vLLM
MODEL_ID      = "google/gemma-4-31B-it"      # ID exact sur le serveur (voir /v1/models)
API_KEY       = "token-local"                # Valeur arbitraire pour vLLM local

DATA_DIR      = "./cluster_data"             # Dossier des données pré-exportées
RESULTS_DIR   = "./results"                  # Répertoire de sortie

# Paramètres LLM
TEMPERATURE   = 0.0        # Déterministe pour reproductibilité
MAX_TOKENS_M1 = 512
MAX_TOKENS_M2 = 768
MAX_TOKENS_M4 = 256
MAX_TOKENS_M6 = 1024

# Troncature du texte de décision pour M6 (caractères)
MAX_DECISION_CHARS = 4000

In [ ]:
import json
import os
import time
import re
from pathlib import Path
from typing import Any, Literal, Optional
from datetime import datetime

import pandas as pd
from pydantic import BaseModel, Field, field_validator
from openai import OpenAI

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import seaborn as sns
    MATPLOTLIB_OK = True
    plt.rcParams['figure.dpi'] = 120
    sns.set_theme(style="whitegrid", palette="muted")
except ImportError:
    MATPLOTLIB_OK = False
    print("[WARN] matplotlib/seaborn non disponibles — graphiques désactivés")

Path(RESULTS_DIR).mkdir(exist_ok=True)

# ── Timestamp du run ──────────────────────────────────────────────────
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Run ID : {RUN_ID}")
print(f"Résultats dans : {RESULTS_DIR}/")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SCHÉMAS PYDANTIC (définis inline — notebook auto-contenu)
# ═══════════════════════════════════════════════════════════════════════

class M1Output(BaseModel):
    """Sortie attendue pour Module 1 — Principe QA."""
    reponse: str = Field(description="Réponse juridique complète en 2-4 paragraphes")
    articles_cites: list[str] = Field(description="Articles de loi cités (ex: 'art. 1641 C. civ.')")
    jurisprudence_citee: list[str] = Field(description="Arrêts cités (ex: 'Cass. soc., 15 janv. 2020')")
    niveau_certitude: Literal["élevé", "moyen", "faible"] = Field(description="Niveau de certitude de la réponse")


class M2Output(BaseModel):
    """Sortie attendue pour Module 2 — Applied QA (avec contexte client)."""
    reponse: str = Field(description="Analyse juridique spécifique à la situation")
    articles_cites: list[str] = Field(description="Articles de loi cités")
    jurisprudence_citee: list[str] = Field(description="Arrêts cités")
    risques: str = Field(description="Risques juridiques identifiés pour le client")
    strategie: str = Field(description="Recommandation pratique (action à mener)")


class M4Output(BaseModel):
    """Sortie attendue pour Module 4 — QCM retrieval."""
    reponse: Literal["A", "B", "C", "D"] = Field(description="Lettre de la bonne réponse")
    justification: str = Field(description="Explication juridique du choix")


class M6Output(BaseModel):
    """Sortie attendue pour Module 6 — Interprétation d'arrêt (7 dimensions)."""
    camp_in_decision: str = Field(description="Qui représente la position du client dans cette décision ?")
    sens_arret: Literal[
        "cassation", "cassation_partielle", "rejet",
        "confirmation", "infirmation", "infirmation_partielle",
        "accueil", "deboute", "non_lieu", "autre"
    ] = Field(description="Dispositif global de la décision")
    is_favorable: bool = Field(description="La décision est-elle favorable au client actuel ?")
    dispositif_summary: str = Field(description="Résumé de ce que décide la cour (2-4 phrases)")
    relevance: float = Field(ge=0.0, le=1.0, description="Score de pertinence pour le dossier (0=inutile, 1=décisif)")
    principles_extracted: list[str] = Field(description="Principes de droit utiles pour le dossier (liste)")
    transfer_reasoning: str = Field(description="Comment cette décision s'applique au dossier du client (1 paragraphe)")


print("Schémas Pydantic chargés.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# VÉRIFICATION DU SERVEUR VLLM
# ═══════════════════════════════════════════════════════════════════════

client = OpenAI(base_url=VLLM_BASE_URL, api_key=API_KEY)

try:
    models = client.models.list()
    model_ids = [m.id for m in models.data]
    print(f"Serveur vLLM OK — modèles disponibles : {model_ids}")

    # Vérifier que le modèle configuré est présent
    if MODEL_ID not in model_ids:
        print(f"[WARN] MODEL_ID='{MODEL_ID}' absent du serveur.")
        print(f"       Modèles disponibles : {model_ids}")
        print(f"       Mise à jour de MODEL_ID → '{model_ids[0]}'")
        MODEL_ID = model_ids[0]

    print(f"\nModèle actif : {MODEL_ID}")

    # Test rapide
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": "Répondez uniquement : OK"}],
        max_tokens=10,
        temperature=0.0,
    )
    latency = time.time() - t0
    print(f"Test LLM → '{resp.choices[0].message.content.strip()}' ({latency:.2f}s)")

except Exception as e:
    print(f"[ERREUR] Impossible de joindre le serveur vLLM : {e}")
    print(f"         Vérifiez que setup_cluster.sh a bien démarré le serveur sur {VLLM_BASE_URL}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# FONCTIONS UTILITAIRES
# ═══════════════════════════════════════════════════════════════════════

def call_llm_structured(
    system_prompt: str,
    user_prompt: str,
    output_model: type[BaseModel],
    max_tokens: int = 512,
) -> tuple[BaseModel | None, dict]:
    """
    Appel vLLM avec sortie JSON structurée (OpenAI response_format).
    Retourne (output_pydantic, meta) où meta contient les stats de l'appel.
    """
    t0 = time.time()
    meta = {"latency_s": None, "tokens_used": None, "error": None}

    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=max_tokens,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": output_model.__name__,
                    "schema": output_model.model_json_schema(),
                    "strict": False,
                },
            },
        )
        raw = response.choices[0].message.content
        meta["latency_s"] = round(time.time() - t0, 2)
        meta["tokens_used"] = response.usage.completion_tokens if response.usage else None

        # Parsing robuste : extraire le JSON même si le modèle ajoute du texte
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            raw = json_match.group(0)

        result = output_model.model_validate_json(raw)
        return result, meta

    except Exception as e:
        meta["latency_s"] = round(time.time() - t0, 2)
        meta["error"] = str(e)
        return None, meta


def keyword_recall(text: str, keywords: list[str]) -> float:
    """Fraction des mots-clés gold trouvés dans le texte de réponse."""
    if not keywords:
        return 0.0
    text_lower = text.lower()
    found = sum(1 for kw in keywords if kw.lower() in text_lower)
    return found / len(keywords)


def article_recall(cited: list[str], gold: list[str]) -> float:
    """Fraction des articles gold cités dans la réponse."""
    if not gold:
        return 1.0  # pas de gold = pas de pénalité
    cited_str = " ".join(cited).lower()
    found = 0
    for art in gold:
        # Chercher les mots-clés de l'article (code + numéro)
        parts = [p for p in re.split(r'[ ,.-]', art.lower()) if len(p) >= 3]
        if any(p in cited_str for p in parts):
            found += 1
    return found / len(gold)


def principles_keyword_overlap(predicted: list[str], gold: list[str]) -> float:
    """Overlap des mots-clés entre les principes prédits et gold."""
    if not gold:
        return 0.0
    stopwords = {"le", "la", "les", "de", "du", "des", "en", "et", "ou",
                 "un", "une", "il", "est", "que", "qui", "sur", "par",
                 "à", "au", "avec", "pour", "pas", "ne", "se"}

    def extract_kw(texts):
        words = set()
        for t in texts:
            for w in re.findall(r'[a-zA-Zéèêëàâùûüçœî]{4,}', t.lower()):
                if w not in stopwords:
                    words.add(w)
        return words

    pred_kw = extract_kw(predicted)
    gold_kw = extract_kw(gold)
    if not pred_kw and not gold_kw:
        return 0.0
    return len(pred_kw & gold_kw) / len(pred_kw | gold_kw)


def format_decision_for_prompt(decision: dict, max_chars: int = MAX_DECISION_CHARS) -> str:
    """Formate le texte de la décision pour le prompt (tronqué, sections structurées si dispo)."""
    s = decision.get("structure") or {}
    parts = []

    header = (
        f"Juridiction : {decision.get('juridiction', '?')} — {decision.get('chambre', '')}\n"
        f"Date : {decision.get('date', '?')} | N° : {decision.get('numero', '?')}\n"
    )
    parts.append(header)

    if s.get("faits"):
        parts.append(f"[FAITS]\n{s['faits'][:600]}")
    if s.get("motifs"):
        parts.append(f"[MOTIFS]\n{s['motifs'][:2200]}")
    if s.get("dispositif"):
        parts.append(f"[DISPOSITIF]\n{s['dispositif'][:600]}")

    if not parts[1:]:  # pas de structure — prendre le texte brut
        raw = decision.get("full_text", "")
        parts.append(raw[:max_chars])

    text = "\n\n".join(parts)
    if len(text) > max_chars:
        text = text[:max_chars] + "\n[...texte tronqué...]"
    return text


print("Fonctions utilitaires chargées.")

---
## M1 — Principe QA
**Format :** Question abstraite de droit → réponse avec articles + JP  
**Données :** 10 questions depuis Les-Audits-Affaires (ou fallback statique)  
**Métriques :** `keyword_recall`, `article_recall`

In [ ]:
# Chargement des questions M1
m1_path = Path(DATA_DIR) / "m1_sample.json"

if m1_path.exists():
    m1_cases = json.loads(m1_path.read_text(encoding="utf-8"))
    print(f"M1 : {len(m1_cases)} questions chargées depuis {m1_path}")
else:
    print(f"[WARN] {m1_path} introuvable — données M1 non disponibles.")
    print("       Exécutez prepare_cluster_data.py en local puis transférez cluster_data/")
    m1_cases = []

# Aperçu
if m1_cases:
    pd.DataFrame(m1_cases)[["id", "specialisation", "question"]].head(5)

In [ ]:
M1_SYSTEM_PROMPT = """Tu es un juriste français expert. Tu réponds aux questions juridiques avec précision et pédagogie.
Tu cites TOUJOURS les textes applicables (codes, articles) et la jurisprudence pertinente.
Ton niveau de certitude reflète l'état du droit : évite les fausses certitudes sur les questions débattues.
Réponds UNIQUEMENT avec le JSON demandé, sans texte avant ou après."""


def build_m1_user_prompt(case: dict) -> str:
    return f"""Réponds à la question juridique suivante en JSON (schéma M1Output) :

QUESTION : {case['question']}

Spécialisation du domaine : {case.get('specialisation', 'non précisée')}"""


print("Prompts M1 définis.")

In [ ]:
m1_results = []

for case in tqdm(m1_cases, desc="M1 — Principe QA"):
    output, meta = call_llm_structured(
        system_prompt=M1_SYSTEM_PROMPT,
        user_prompt=build_m1_user_prompt(case),
        output_model=M1Output,
        max_tokens=MAX_TOKENS_M1,
    )

    if output is None:
        row = {"id": case["id"], "error": meta["error"], **meta}
    else:
        kw_rec = keyword_recall(output.reponse, case.get("gold_keywords", []))
        art_rec = article_recall(output.articles_cites, case.get("gold_articles", []))
        row = {
            "id": case["id"],
            "specialisation": case.get("specialisation", ""),
            "question_short": case["question"][:60] + "…",
            "keyword_recall": round(kw_rec, 3),
            "article_recall": round(art_rec, 3),
            "score_m1": round((kw_rec + art_rec) / 2, 3),
            "niveau_certitude": output.niveau_certitude,
            "n_articles_cites": len(output.articles_cites),
            "n_jp_citee": len(output.jurisprudence_citee),
            "reponse_preview": output.reponse[:120] + "…",
            "latency_s": meta["latency_s"],
        }
    m1_results.append(row)

df_m1 = pd.DataFrame(m1_results)
print(f"M1 terminé — {len(df_m1)} réponses")

In [ ]:
# Affichage résultats M1
print("═" * 60)
print("RÉSULTATS M1 — Principe QA")
print("═" * 60)

display_cols = [c for c in ["id", "specialisation", "keyword_recall", "article_recall", "score_m1", "niveau_certitude", "latency_s"] if c in df_m1.columns]
print(df_m1[display_cols].to_string(index=False))

if "score_m1" in df_m1.columns:
    print(f"\nScore M1 moyen : {df_m1['score_m1'].mean():.3f}")
    print(f"Keyword recall : {df_m1['keyword_recall'].mean():.3f}")
    print(f"Article recall : {df_m1['article_recall'].mean():.3f}")
    print(f"Latence moyenne : {df_m1['latency_s'].mean():.2f}s")

---
## M2 — Applied QA (avec contexte client)
**Format :** Dossier client + question → analyse contextualisée (articles + risques + stratégie)  
**Données :** 3 cas construits manuellement  
**Métriques :** `article_recall`, `strategy_present`

In [ ]:
# ── Cas M2 (hardcodés) ───────────────────────────────────────────────

M2_CASES = [
    {
        "id": "M2-S-001",
        "specialisation": "Droit Social",
        "difficulty": "easy",
        "case_summary": (
            "Salarié cadre avec 5 ans d'ancienneté. Son contrat comprend une clause de non-concurrence "
            "valable 2 ans sur l'ensemble du territoire national, sans aucune contrepartie financière mentionnée. "
            "Il vient de démissionner et veut rejoindre un concurrent direct dans la même ville."
        ),
        "client_position": "defense",
        "key_facts": [
            "clause de non-concurrence dans le contrat",
            "aucune contrepartie financière prévue",
            "périmètre national (territoire entier)",
            "durée 2 ans",
            "poste visé = concurrent direct",
        ],
        "question": "L'ancien employeur peut-il contraindre le salarié à respecter cette clause de non-concurrence ?",
        "gold_articles": ["L. 1121-1 Code du travail"],
        "gold_keywords": ["contrepartie", "nulle", "non-concurrence", "invalide"],
    },
    {
        "id": "M2-S-002",
        "specialisation": "Droit Civil",
        "difficulty": "medium",
        "case_summary": (
            "Particulier a acheté il y a 8 mois une voiture d'occasion auprès d'un vendeur professionnel. "
            "La boîte automatique vient de lâcher (coût de réparation : 4 500 €). Le garagiste confirme "
            "que le vice était antérieur à la vente (usure anormale interne invisible à l'achat). "
            "Le vendeur refuse tout remboursement et invoque l'état de la voiture au moment de la vente."
        ),
        "client_position": "demande",
        "key_facts": [
            "achat il y a 8 mois (dans le délai de prescription)",
            "vice antérieur à la vente (attestation garagiste)",
            "vendeur professionnel (présomption de connaissance du vice)",
            "vice invisible à l'inspection normale",
            "préjudice : 4 500 €",
        ],
        "question": "Quels sont les recours de l'acheteur contre le vendeur professionnel ?",
        "gold_articles": ["1641 Code civil", "1643 Code civil", "1644 Code civil"],
        "gold_keywords": ["vice caché", "action rédhibitoire", "estimatoire", "dommages-intérêts", "professionnel"],
    },
    {
        "id": "M2-S-003",
        "specialisation": "Droit Pénal",
        "difficulty": "hard",
        "case_summary": (
            "Commerçant qui a surpris un individu en train de voler dans son magasin. "
            "Il l'a physiquement retenu (bras dans le dos, pendant 12 minutes) "
            "jusqu'à l'arrivée de la police. L'individu s'est légèrement blessé au poignet. "
            "Il porte plainte pour séquestration et violences. Le commerçant est poursuivi."
        ),
        "client_position": "defense",
        "key_facts": [
            "flagrant délit de vol constaté",
            "rétention physique brève (12 minutes, jusqu'à la police)",
            "blessure légère (poignet)",
            "plainte double : séquestration + violences",
        ],
        "question": (
            "Le commerçant peut-il invoquer une cause d'irresponsabilité pour la rétention physique "
            "et les blessures légères occasionnées ?"
        ),
        "gold_articles": ["122-5 Code pénal", "122-7 Code pénal", "73 Code de procédure pénale"],
        "gold_keywords": ["légitime défense", "état de nécessité", "citoyen", "appréhension", "flagrant délit"],
    },
]

print(f"M2 : {len(M2_CASES)} cas chargés")
for c in M2_CASES:
    print(f"  {c['id']} — {c['specialisation']} ({c['difficulty']})")

In [ ]:
M2_SYSTEM_PROMPT = """Tu es un juriste français expert spécialisé en conseil client.
Tu analyses des situations juridiques précises et fournis des réponses opérationnelles.
Tes réponses incluent les textes applicables, les risques et une stratégie concrète.
Réponds UNIQUEMENT avec le JSON demandé, sans texte avant ou après."""


def build_m2_user_prompt(case: dict) -> str:
    facts = "\n".join(f"  - {f}" for f in case["key_facts"])
    pos = "demandeur" if case["client_position"] == "demande" else "défendeur"
    return f"""Analyse la situation juridique suivante (JSON schéma M2Output) :

DOMAINE : {case['specialisation']}
POSITION CLIENT : {pos}

RÉSUMÉ DU DOSSIER :
{case['case_summary']}

FAITS CLÉS :
{facts}

QUESTION : {case['question']}"""


m2_results = []

for case in tqdm(M2_CASES, desc="M2 — Applied QA"):
    output, meta = call_llm_structured(
        system_prompt=M2_SYSTEM_PROMPT,
        user_prompt=build_m2_user_prompt(case),
        output_model=M2Output,
        max_tokens=MAX_TOKENS_M2,
    )

    if output is None:
        row = {"id": case["id"], "error": meta["error"]}
    else:
        art_rec = article_recall(output.articles_cites, case.get("gold_articles", []))
        kw_rec = keyword_recall(output.reponse, case.get("gold_keywords", []))
        has_strategy = len(output.strategie) > 20
        row = {
            "id": case["id"],
            "specialisation": case["specialisation"],
            "difficulty": case["difficulty"],
            "keyword_recall": round(kw_rec, 3),
            "article_recall": round(art_rec, 3),
            "strategy_present": has_strategy,
            "score_m2": round((kw_rec + art_rec + int(has_strategy)) / 3, 3),
            "latency_s": meta["latency_s"],
            "reponse_preview": output.reponse[:120] + "…",
            "strategie_preview": output.strategie[:80] + "…",
        }
    m2_results.append(row)

df_m2 = pd.DataFrame(m2_results)
print(f"M2 terminé — {len(df_m2)} réponses")

display_cols = [c for c in ["id", "specialisation", "difficulty", "keyword_recall", "article_recall", "strategy_present", "score_m2", "latency_s"] if c in df_m2.columns]
print(df_m2[display_cols].to_string(index=False))
if "score_m2" in df_m2.columns:
    print(f"\nScore M2 moyen : {df_m2['score_m2'].mean():.3f}")

---
## M3 — Raisonnement multi-saut ⛔ SKIP
**Raison :** Nécessite le Knowledge Graph v1 (citations entre décisions) — non construit à ce stade.  
**Prévu :** Phase B3 du plan de construction.

In [ ]:
m3_status = {"module": "M3", "status": "SKIP", "reason": "KG v1 non disponible", "score": None}
print(f"M3 — Raisonnement multi-saut : SKIP ({m3_status['reason']})")

---
## M4 — QCM Retrieval
**Format :** Question à choix multiples (A/B/C/D) — 5 spécialisations couvertes  
**Données :** 5 QCM construits manuellement depuis le droit positif  
**Métriques :** `accuracy` (simple, automatique)

In [ ]:
M4_QCM = [
    {
        "id": "M4-QCM-001",
        "specialisation": "Droit Social",
        "difficulty": "easy",
        "question": "La clause de non-concurrence dans un contrat de travail est valide si elle comporte OBLIGATOIREMENT :",
        "options": {
            "A": "Une limite géographique uniquement",
            "B": "Une limite dans le temps, dans l'espace, une contrepartie financière et la protection d'intérêts légitimes",
            "C": "Une limite temporelle uniquement",
            "D": "L'accord écrit du salarié au moment de la rupture",
        },
        "gold_answer": "B",
        "gold_article": "Cass. soc., 10 juill. 2002 / L. 1121-1 C. trav.",
    },
    {
        "id": "M4-QCM-002",
        "specialisation": "Droit Civil",
        "difficulty": "easy",
        "question": "En matière de vices cachés (art. 1641 C. civ.), l'action de l'acheteur se prescrit en :",
        "options": {
            "A": "1 an à compter de la découverte",
            "B": "2 ans à compter de la découverte du vice",
            "C": "5 ans à compter de la vente",
            "D": "10 ans à compter de la livraison du bien",
        },
        "gold_answer": "B",
        "gold_article": "1648 Code civil",
    },
    {
        "id": "M4-QCM-003",
        "specialisation": "Droit Pénal",
        "difficulty": "medium",
        "question": "La légitime défense (art. 122-5 C. pén.) suppose TOUJOURS que la riposte soit :",
        "options": {
            "A": "Immédiate et préméditée contre l'agresseur identifié",
            "B": "Nécessaire, simultanée et proportionnée à l'agression réelle ou imminente",
            "C": "Exercée uniquement par les forces de l'ordre",
            "D": "Autorisée uniquement en cas d'agression armée",
        },
        "gold_answer": "B",
        "gold_article": "122-5 Code pénal",
    },
    {
        "id": "M4-QCM-004",
        "specialisation": "Droit Commercial",
        "difficulty": "medium",
        "question": "La rupture brutale des relations commerciales établies (art. L. 442-1, II C. com.) suppose :",
        "options": {
            "A": "Obligatoirement l'existence d'un contrat écrit entre les parties",
            "B": "Une rupture sans préavis écrit raisonnable eu égard à la durée de la relation",
            "C": "Un chiffre d'affaires annuel minimum prouvé par factures",
            "D": "Une rupture dans le cadre d'une procédure collective de l'auteur",
        },
        "gold_answer": "B",
        "gold_article": "L. 442-1 II Code de commerce",
    },
    {
        "id": "M4-QCM-005",
        "specialisation": "Droit de la Famille",
        "difficulty": "easy",
        "question": "La prestation compensatoire (art. 270-271 C. civ.) est due :",
        "options": {
            "A": "Uniquement par le conjoint qui a initié la procédure de divorce",
            "B": "Par le conjoint le plus aisé, pour compenser la disparité des conditions de vie créée par la rupture",
            "C": "Uniquement dans le cadre du divorce pour faute",
            "D": "Par le conjoint qui a la garde exclusive des enfants",
        },
        "gold_answer": "B",
        "gold_article": "270-271 Code civil",
    },
]

print(f"M4 : {len(M4_QCM)} QCM chargés")
for q in M4_QCM:
    print(f"  {q['id']} — {q['specialisation']} | Gold: {q['gold_answer']}")

In [ ]:
M4_SYSTEM_PROMPT = """Tu es un juriste français expert. Tu réponds à des questions à choix multiples de droit français.
Tu choisis la SEULE réponse correcte parmi A, B, C ou D et tu justifies ton choix.
Réponds UNIQUEMENT avec le JSON demandé, sans texte avant ou après."""


def build_m4_user_prompt(qcm: dict) -> str:
    opts = "\n".join(f"{k}) {v}" for k, v in qcm["options"].items())
    return f"""Question à choix multiples (JSON schéma M4Output) :

DOMAINE : {qcm['specialisation']}

QUESTION : {qcm['question']}

{opts}"""


m4_results = []

for qcm in tqdm(M4_QCM, desc="M4 — QCM"):
    output, meta = call_llm_structured(
        system_prompt=M4_SYSTEM_PROMPT,
        user_prompt=build_m4_user_prompt(qcm),
        output_model=M4Output,
        max_tokens=MAX_TOKENS_M4,
    )

    if output is None:
        row = {"id": qcm["id"], "error": meta["error"]}
    else:
        correct = output.reponse == qcm["gold_answer"]
        row = {
            "id": qcm["id"],
            "specialisation": qcm["specialisation"],
            "difficulty": qcm["difficulty"],
            "predicted": output.reponse,
            "gold": qcm["gold_answer"],
            "correct": correct,
            "score_m4": 1.0 if correct else 0.0,
            "justification_preview": output.justification[:100] + "…",
            "latency_s": meta["latency_s"],
        }
    m4_results.append(row)

df_m4 = pd.DataFrame(m4_results)
print(f"M4 terminé — {len(df_m4)} réponses")

display_cols = [c for c in ["id", "specialisation", "predicted", "gold", "correct", "latency_s"] if c in df_m4.columns]
print(df_m4[display_cols].to_string(index=False))
if "score_m4" in df_m4.columns:
    acc = df_m4["correct"].mean()
    print(f"\nAccuracy M4 : {acc:.1%} ({df_m4['correct'].sum()}/{len(df_m4)}")

---
## M5 — Temporalité ⛔ SKIP
**Raison :** Nécessite un KG avec versioning temporel des articles — non construit à ce stade.  
**Prévu :** Phase B3 du plan de construction.

In [ ]:
m5_status = {"module": "M5", "status": "SKIP", "reason": "Versioning temporel des articles non disponible", "score": None}
print(f"M5 — Temporalité : SKIP ({m5_status['reason']})")

---
## M6 — Interprétation d'arrêt contextualisée ⭐
**Format :** Dossier client + décision fournie → analyse à 7 dimensions  
**Données :** 5 cas gold standard annotés manuellement  
**Métriques :** `is_favorable_acc`, `sens_arret_acc`, `relevance_mae`, `principles_overlap`  
**Cas piège :** M6-JP-004 (cassation procédurale — ne doit PAS être is_favorable=True)

In [ ]:
m6_path = Path(DATA_DIR) / "m6_gold_cases.json"

if m6_path.exists():
    m6_cases = json.loads(m6_path.read_text(encoding="utf-8"))
    print(f"M6 : {len(m6_cases)} cas gold chargés depuis {m6_path}")
    for c in m6_cases:
        fav = "✓ favorable" if c['gold']['is_favorable'] else "✗ défav."
        trap = " [PIÈGE]" if c['difficulty'] == 'trap' else ""
        print(f"  {c['id']} — {c['specialisation']} ({c['difficulty']}) | {c['gold']['sens_arret']} | {fav}{trap}")
else:
    print(f"[WARN] {m6_path} introuvable.")
    print("       Exécutez prepare_cluster_data.py en local puis transférez cluster_data/")
    m6_cases = []

In [ ]:
M6_SYSTEM_PROMPT = """Tu es un avocat expert en droit français. Tu analyses des décisions de justice \
dans le contexte d'un dossier client précis.
Tu évalues avec rigueur si la décision est utile pour le client, en distinguant :
- Ce que décide réellement la cour (fond vs procédure)
- Si la décision est vraiment favorable au client (attention aux cassations procédurales)
- Les principes de droit directement transférables au dossier
Réponds UNIQUEMENT avec le JSON demandé, sans texte avant ou après."""


def build_m6_user_prompt(case: dict) -> str:
    c = case["case"]
    facts = "\n".join(f"  - {f}" for f in c["key_facts"])
    pos = "demandeur" if c["client_position"] == "demande" else "défendeur"
    decision_text = format_decision_for_prompt(case["decision"])

    return f"""Analyse la décision de justice fournie pour le dossier client ci-dessous (JSON schéma M6Output) :

━━━ DOSSIER CLIENT ━━━
Spécialisation : {case['specialisation']}
Position client : {pos}

Résumé : {c['case_summary']}

Faits clés :
{facts}

Question : {case['question']}

━━━ DÉCISION DE JUSTICE ━━━
{decision_text}"""


m6_results = []

for case in tqdm(m6_cases, desc="M6 — Interprétation d'arrêt"):
    output, meta = call_llm_structured(
        system_prompt=M6_SYSTEM_PROMPT,
        user_prompt=build_m6_user_prompt(case),
        output_model=M6Output,
        max_tokens=MAX_TOKENS_M6,
    )

    gold = case["gold"]

    if output is None:
        row = {
            "id": case["id"],
            "specialisation": case["specialisation"],
            "difficulty": case["difficulty"],
            "error": meta["error"],
        }
    else:
        is_fav_ok    = output.is_favorable == gold["is_favorable"]
        sens_ok      = output.sens_arret   == gold["sens_arret"]
        rel_mae      = abs(output.relevance - gold["relevance"])
        princ_olap   = principles_keyword_overlap(
            output.principles_extracted, gold["principles_extracted"]
        )
        # Score composite (is_favorable pondéré 2x car critique)
        composite = (2 * int(is_fav_ok) + int(sens_ok) + (1 - rel_mae) + princ_olap) / 5

        row = {
            "id": case["id"],
            "specialisation": case["specialisation"],
            "difficulty": case["difficulty"],
            # Résultats gold
            "gold_is_favorable": gold["is_favorable"],
            "gold_sens_arret":   gold["sens_arret"],
            "gold_relevance":    gold["relevance"],
            # Prédictions modèle
            "pred_is_favorable": output.is_favorable,
            "pred_sens_arret":   output.sens_arret,
            "pred_relevance":    round(output.relevance, 2),
            # Métriques
            "is_favorable_ok":   is_fav_ok,
            "sens_arret_ok":     sens_ok,
            "relevance_mae":     round(rel_mae, 3),
            "principles_overlap": round(princ_olap, 3),
            "score_m6":          round(composite, 3),
            "latency_s":         meta["latency_s"],
            # Aperçu
            "pred_dispositif_preview": output.dispositif_summary[:100] + "…",
        }
    m6_results.append(row)

df_m6 = pd.DataFrame(m6_results)
print(f"M6 terminé — {len(df_m6)} réponses")

In [ ]:
print("═" * 80)
print("RÉSULTATS M6 — Interprétation d'arrêt")
print("═" * 80)

display_cols = [c for c in [
    "id", "difficulty",
    "gold_is_favorable", "pred_is_favorable", "is_favorable_ok",
    "gold_sens_arret",   "pred_sens_arret",   "sens_arret_ok",
    "gold_relevance",    "pred_relevance",    "relevance_mae",
    "principles_overlap", "score_m6", "latency_s"
] if c in df_m6.columns]

print(df_m6[display_cols].to_string(index=False))

if "score_m6" in df_m6.columns:
    print(f"""
Métriques M6 :
  is_favorable accuracy : {df_m6['is_favorable_ok'].mean():.1%}  (métrique cœur — piège sur M6-JP-004)
  sens_arret accuracy   : {df_m6['sens_arret_ok'].mean():.1%}
  relevance MAE         : {df_m6['relevance_mae'].mean():.3f}  (0=parfait)
  principles overlap    : {df_m6['principles_overlap'].mean():.3f}  (Jaccard kw)
  score M6 composite    : {df_m6['score_m6'].mean():.3f}
""")

    # Identifier si le modèle a bien gèré le cas piège
    trap = df_m6[df_m6["difficulty"] == "trap"]
    if not trap.empty:
        trap_ok = trap["is_favorable_ok"].all()
        status = "✓ PIÈGE DÉTECTÉ" if trap_ok else "✗ PIÈGE RATÉ (LLM naïf)"
        print(f"  Cas piège (M6-JP-004) : {status}")

---
## Résultats consolidés — B1 (LLM seul, zero-shot)
Agrégation des scores par module + export CSV

In [ ]:
# ── Tableau de synthèse ───────────────────────────────────────────────
summary_rows = []

if not df_m1.empty and "score_m1" in df_m1.columns:
    summary_rows.append({
        "module": "M1 — Principe QA",
        "n_items": len(df_m1),
        "main_metric": "score (kw+art recall) / 2",
        "score": round(df_m1["score_m1"].mean(), 3),
        "latency_mean_s": round(df_m1["latency_s"].mean(), 2),
        "status": "OK",
    })

if not df_m2.empty and "score_m2" in df_m2.columns:
    summary_rows.append({
        "module": "M2 — Applied QA",
        "n_items": len(df_m2),
        "main_metric": "score (kw+art+strategy) / 3",
        "score": round(df_m2["score_m2"].mean(), 3),
        "latency_mean_s": round(df_m2["latency_s"].mean(), 2),
        "status": "OK",
    })

summary_rows.append({"module": "M3 — Multi-saut", "n_items": 0, "main_metric": "—", "score": None, "latency_mean_s": None, "status": "SKIP (KG v1 manquant)"})

if not df_m4.empty and "score_m4" in df_m4.columns:
    summary_rows.append({
        "module": "M4 — QCM",
        "n_items": len(df_m4),
        "main_metric": "accuracy",
        "score": round(df_m4["score_m4"].mean(), 3),
        "latency_mean_s": round(df_m4["latency_s"].mean(), 2),
        "status": "OK",
    })

summary_rows.append({"module": "M5 — Temporalité", "n_items": 0, "main_metric": "—", "score": None, "latency_mean_s": None, "status": "SKIP (versioning temporel manquant)"})

if not df_m6.empty and "score_m6" in df_m6.columns:
    summary_rows.append({
        "module": "M6 — Interprétation d'arrêt ⭐",
        "n_items": len(df_m6),
        "main_metric": "composite (is_fav×2 + sens + (1-mae) + overlap) / 5",
        "score": round(df_m6["score_m6"].mean(), 3),
        "latency_mean_s": round(df_m6["latency_s"].mean(), 2),
        "status": "OK",
    })

df_summary = pd.DataFrame(summary_rows)

print("═" * 80)
print(f"BENCHMARK M1–M6 — Configuration B1 (LLM seul) — Modèle : {MODEL_ID}")
print(f"Run : {RUN_ID}")
print("═" * 80)
print(df_summary[["module", "n_items", "score", "latency_mean_s", "status"]].to_string(index=False))

# Score global (moyenne pondérée des modules disponibles, M6 pondéré ×2)
scores_available = [(r["score"], 2 if "M6" in r["module"] else 1) for r in summary_rows if r["score"] is not None]
if scores_available:
    total_w = sum(w for _, w in scores_available)
    weighted_score = sum(s * w for s, w in scores_available) / total_w
    print(f"\nScore global pondéré (M6 ×2) : {weighted_score:.3f}")
    print(f"(M1:×1, M2:×1, M4:×1, M6:×2 — M3/M5 exclus)")

In [ ]:
# ── Export CSV ────────────────────────────────────────────────────────
results_base = Path(RESULTS_DIR) / f"{RUN_ID}_{MODEL_ID.replace('/', '_')}"

exports = [
    (df_m1,      f"{results_base}_M1.csv"),
    (df_m2,      f"{results_base}_M2.csv"),
    (df_m4,      f"{results_base}_M4.csv"),
    (df_m6,      f"{results_base}_M6.csv"),
    (df_summary, f"{results_base}_summary.csv"),
]

for df, path in exports:
    if not df.empty:
        df.to_csv(path, index=False, sep=";", encoding="utf-8-sig")
        print(f"Exporté : {path}")

# Export JSON consolidé
consolidated = {
    "run_id": RUN_ID,
    "model_id": MODEL_ID,
    "config": "B1_zero_shot",
    "summary": df_summary.to_dict(orient="records"),
    "m1_detail": df_m1.to_dict(orient="records") if not df_m1.empty else [],
    "m2_detail": df_m2.to_dict(orient="records") if not df_m2.empty else [],
    "m4_detail": df_m4.to_dict(orient="records") if not df_m4.empty else [],
    "m6_detail": df_m6.to_dict(orient="records") if not df_m6.empty else [],
}
json_path = f"{results_base}_full.json"
Path(json_path).write_text(json.dumps(consolidated, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Exporté : {json_path}")

In [ ]:
if not MATPLOTLIB_OK:
    print("matplotlib non disponible — graphiques ignorés")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"Benchmark M1–M6 — B1 zero-shot\nModèle : {MODEL_ID}", fontsize=12, fontweight="bold")

    # ── Graphique 1 : Scores par module ──────────────────────────────────
    ax = axes[0]
    scores_df = df_summary[df_summary["score"].notna()].copy()
    colors = ["#2196F3" if "M6" not in r else "#FF9800" for r in scores_df["module"]]
    bars = ax.barh(scores_df["module"].str.replace(" ⭐", ""), scores_df["score"], color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Score")
    ax.set_title("Scores par module")
    for bar, score in zip(bars, scores_df["score"]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{score:.3f}", va="center", fontsize=9)
    ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="seuil 0.5")
    ax.legend(fontsize=8)

    # ── Graphique 2 : M6 — Dimensions détaillées ─────────────────────────
    ax = axes[1]
    if not df_m6.empty and "is_favorable_ok" in df_m6.columns:
        m6_metrics = {
            "is_favorable": df_m6["is_favorable_ok"].mean(),
            "sens_arret":   df_m6["sens_arret_ok"].mean(),
            "1 - rel_mae":  1 - df_m6["relevance_mae"].mean(),
            "principles":   df_m6["principles_overlap"].mean(),
        }
        metric_names = list(m6_metrics.keys())
        metric_vals  = list(m6_metrics.values())
        bar_colors = ["#4CAF50" if v >= 0.6 else "#F44336" if v < 0.4 else "#FF9800" for v in metric_vals]
        ax.bar(metric_names, metric_vals, color=bar_colors)
        ax.set_ylim(0, 1)
        ax.set_ylabel("Score")
        ax.set_title("M6 — 4 dimensions automatiques")
        for i, v in enumerate(metric_vals):
            ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=10, fontweight="bold")
        ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
    else:
        ax.text(0.5, 0.5, "Données M6\nnon disponibles", ha="center", va="center", transform=ax.transAxes)

    # ── Graphique 3 : M6 — is_favorable par cas (piège mis en évidence) ──
    ax = axes[2]
    if not df_m6.empty and "gold_is_favorable" in df_m6.columns:
        x = range(len(df_m6))
        gold_vals = df_m6["gold_is_favorable"].astype(int)
        pred_vals = df_m6["pred_is_favorable"].astype(int)
        labels    = df_m6["id"].str.replace("M6-JP-", "")
        colors_gold = ["#4CAF50" if v else "#F44336" for v in df_m6["gold_is_favorable"]]
        colors_pred = ["#4CAF50" if v else "#F44336" for v in df_m6["pred_is_favorable"]]

        ax.scatter([i - 0.15 for i in x], gold_vals, color=colors_gold, s=120, marker="o", label="Gold", zorder=3)
        ax.scatter([i + 0.15 for i in x], pred_vals, color=colors_pred, s=120, marker="^", label="Prédit", zorder=3)
        ax.set_xticks(list(x))
        ax.set_xticklabels(labels, rotation=30, fontsize=8)
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["Défav.", "Fav."])
        ax.set_title("M6 — is_favorable\n(◯ gold, △ prédit)")
        ax.legend(fontsize=8)
        ax.set_ylim(-0.3, 1.3)
        # Surligner le cas piège
        trap_idx = df_m6[df_m6["difficulty"] == "trap"].index
        for idx in trap_idx:
            ax.axvspan(idx - 0.4, idx + 0.4, alpha=0.1, color="red")
            ax.text(idx, 1.2, "PIÈGE", ha="center", fontsize=7, color="red")
    else:
        ax.text(0.5, 0.5, "Données M6\nnon disponibles", ha="center", va="center", transform=ax.transAxes)

    plt.tight_layout()
    plot_path = f"{results_base}_plot.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Graphique enregistré : {plot_path}")

---
## Interprétation des résultats

### Ce que ces scores mesurent
- **M1 / M2** : rappel lexical (articles, mots-clés) — métrique proxy, imparfaite (ne mesure pas la justesse réelle)
- **M4** : accuracy exacte — métrique fiable, format QCM
- **M6** : métriques automatiques sur 4 dimensions ; la 5e (`dispositif_summary`) et la 7e (`transfer_reasoning`) nécessitent un LLM-judge (prochaine itération)

### Cas piège M6-JP-004
- Gold : `is_favorable=False`, `relevance=0.20` (cassation purement procédurale)
- Un LLM naïf voit « cassation » → répond `is_favorable=True` → score 0 sur la métrique cœur
- **Si le modèle passe ce test → il comprend la distinction fond/procédure**

### Prochaines étapes
1. Ajouter LLM-as-judge (Judge 1 + Judge 2) pour `dispositif_summary` et `transfer_reasoning`
2. Tester configurations B2 (RAG vectoriel), B4 (RAG + agentique)
3. Comparer avec configuration C (GraphRAG)
4. Étendre à 30-40 cas par spécialisation pour M6 (target final)